Author: Malachy McCaffrey

Create fishnet grid where each AIS point is the centroid of a grid cell. Have it match with GFW grid schema. Convert to UTM Zone 19N.

Steps:
1) Convert gfw vp and afe csv files to projected point feature classes (UTM 19N)
2) Create projected orsted fishnet grid from gfw_vp_id (has most observations so most complete spatial grid within AOI)

## Call ArcPy and Set Working Environment

In [7]:
import arcpy
from arcpy import env
import os

env.workspace = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb"
env.overwriteOutput = True

workspace = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb"

## Step 1: Convert raw data files to point feature classes then project

Convert all csv files from processedAIS folder to project gdb

In [9]:
# define input and output

input_folder = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\processedAIS"
output_gdb = workspace

# define coordinate fields

lon = "Lon"
lat = "Lat"

# define spatial references

wgs1984 = arcpy.SpatialReference(4326)
utm19n = arcpy.SpatialReference(32619)

# for loop

for file in os.listdir(input_folder):
    if file.lower().endswith(".csv"):
        csv_path = os.path.join(input_folder, file)
        fc_name = os.path.splitext(file)[0]
        temp_fc = os.path.join(output_gdb, fc_name)

        projected_fc = os.path.join(
            output_gdb,
            fc_name + "_utm19n"
        )

        arcpy.management.XYTableToPoint(
            in_table=csv_path,
            out_feature_class=temp_fc,
            x_field=lon,
            y_field=lat,
            coordinate_system=wgs1984
            )
        
        arcpy.management.Project(
            temp_fc,
            projected_fc,
            arcpy.SpatialReference(32619)
        )

        count = int(arcpy.management.GetCount(projected_fc)[0])

        print(f"{fc_name} ({count} points)")

print("Conversion and projection complete")

# gfw_afe1_id_sub (3264 points)
# gfw_afe2_id_sub (4617 points) # didn't change after repulling data on 7/17 from 7/14...
# gfw_afe3_id_sub (11187 points)
# gfw_afe_id_sub (19068 points)
# gfw_vp1_fv_sub (23078 points)
# gfw_vp2_fv_sub (28151 points) # didn't change after repulling data on 7/17 from 7/14... why only dev stage 2?
# gfw_vp3_fv_sub (36806 points)
# gfw_vp_fv (137539 points)
# gfw_vp_fv_sub (88035 points)
# gfw_vp_id (416533 points)
# gfw_vp_orstedVessels (55969 points)

gfw_afe1_id_sub (3264 points)
gfw_afe2_id_sub (4617 points)
gfw_afe3_id_sub (11187 points)
gfw_afe_id_sub (19068 points)
gfw_vp1_fv_sub (23078 points)
gfw_vp2_fv_sub (28151 points)
gfw_vp3_fv_sub (36806 points)
gfw_vp_fv (137539 points)
gfw_vp_fv_sub (88035 points)
gfw_vp_id (416533 points)
gfw_vp_orstedVessels (55969 points)
Conversion and projection complete


## Step 2: Create projected fishnet grid

Create grid where each AIS point is the centroid of a grid cell. Have it match with GFW grid schema. Convert to UTM 19N. 

In [11]:
# input point feature class

gfw_vp_pts = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_vesselHotspots_SNE\\GFW_vesselHotspots_SNE.gdb\\gfw_vp_id"

# confirm spatial ref

desc = arcpy.Describe(gfw_vp_pts)
print(desc.spatialReference.name)

GCS_WGS_1984


In [12]:
# create unique point feature class

unique_pts = "gfw_vp_unique_pts"

arcpy.management.CopyFeatures(
    in_features=gfw_vp_pts,
    out_feature_class=unique_pts
)

arcpy.management.DeleteIdentical(
    in_dataset=unique_pts,
    fields=["Lon", "Lat"]
)

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb\\gfw_vp_unique_pts'>

In [14]:
# create empty polygon feature class

AOI_sqGrid = "Orsted_sqGridWGS"

arcpy.management.CreateFeatureclass(
    out_path=env.workspace,
    out_name=AOI_sqGrid,
    geometry_type="POLYGON",
    spatial_reference=gfw_vp_pts
)

# carry cell centroids over

arcpy.management.AddField(AOI_sqGrid, "Lon_C", "DOUBLE")
arcpy.management.AddField(AOI_sqGrid, "Lat_C", "DOUBLE")

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb\\Orsted_sqGridWGS'>

In [15]:
# construct 0.01 degree square grid around each point

half = 0.005  # half of 0.01 degrees

with arcpy.da.SearchCursor(unique_pts, ["SHAPE@XY", "Lon", "Lat"]) as scur, \
     arcpy.da.InsertCursor(AOI_sqGrid, ["SHAPE@", "Lon_C", "Lat_C"]) as icur:

    for (x, y), lon, lat in scur:

        array = arcpy.Array([
            arcpy.Point(x - half, y - half),  # lower left
            arcpy.Point(x - half, y + half),  # upper left
            arcpy.Point(x + half, y + half),  # upper right
            arcpy.Point(x + half, y - half),  # lower right
            arcpy.Point(x - half, y - half)   # close polygon
        ])

        polygon = arcpy.Polygon(array, arcpy.Describe(unique_pts).spatialReference)

        icur.insertRow([polygon, lon, lat])

In [16]:
# validate geometry by calculating polygon centroids

arcpy.management.FeatureToPoint(
    in_features=AOI_sqGrid,
    out_feature_class="grid_centroids",
    point_location="CENTROID"
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\GFW_VesselHotspots_SNE\\\\GFW_VesselHotspots_SNE.gdb\\grid_centroids'>

In [17]:
# use near analysis to measure offset

arcpy.analysis.Near(
    in_features="grid_centroids",
    near_features=unique_pts
) # adds a NEAR_DIST field

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb\\grid_centroids'>

In [18]:
# check for errors

with arcpy.da.SearchCursor("grid_centroids", ["NEAR_DIST"]) as cursor:
    max_dist = max(row[0] for row in cursor)

print(f"Maximum centroid-point distance: {max_dist}") # e-15 value is a safe floating-point artifact

Maximum centroid-point distance: 0.0


Good to move on to UTM 19N projection!

In [19]:
# define inputs and outputs

# AOI_sqGridWGS already defined
# gfw_vp_pts already defined

AOI_sqGrid_utm19n = "Orsted_sqGrid_utm19n" # output polygon
gfw_vp_pts_utm19n = "gfw_vp_pts_utm19n" # output point

In [20]:
# verify source spatial reference

for fc in [AOI_sqGrid, gfw_vp_pts]:
    sr = arcpy.Describe(fc).spatialReference

    print(fc, sr.name, sr.factoryCode)

Orsted_sqGridWGS GCS_WGS_1984 4326
C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_vesselHotspots_SNE\\GFW_vesselHotspots_SNE.gdb\\gfw_vp_id GCS_WGS_1984 4326


In [22]:
# project the polygon grid

arcpy.management.Project(
    in_dataset=AOI_sqGrid,
    out_dataset=AOI_sqGrid_utm19n,
    out_coor_system=utm19n,
    transform_method="", # not required for WGS84 to WGS 84
    preserve_shape="PRESERVE_SHAPE"
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\GFW_VesselHotspots_SNE\\\\GFW_VesselHotspots_SNE.gdb\\Orsted_sqGrid_utm19n'>

In [23]:
# project the point feature class

arcpy.management.Project(
    in_dataset=gfw_vp_pts,
    out_dataset=gfw_vp_pts_utm19n,
    out_coor_system=utm19n,
    transform_method=""
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\GFW_VesselHotspots_SNE\\\\GFW_VesselHotspots_SNE.gdb\\gfw_vp_pts_utm19n'>

In [24]:
# validate geometry by double checking projections

for fc in [AOI_sqGrid_utm19n, gfw_vp_pts_utm19n]:
    sr = arcpy.Describe(fc).spatialReference

    print(fc, sr.name, sr.linearUnitName)

Orsted_sqGrid_utm19n WGS_1984_UTM_Zone_19N Meter
gfw_vp_pts_utm19n WGS_1984_UTM_Zone_19N Meter


In [25]:
##  confirm centroid-to-centroid alignment

# create polygon centroids

arcpy.management.FeatureToPoint(
    in_features=AOI_sqGrid_utm19n,
    out_feature_class="grid_utm_centroids",
    point_location="CENTROID"
)

# measure distance to AIS points

arcpy.analysis.Near(
    in_features="grid_utm_centroids",
    near_features=gfw_vp_pts_utm19n
)

# inspect max offset

with arcpy.da.SearchCursor(
    "grid_utm_centroids", ["NEAR_DIST"]
) as cursor:
    max_dist = max(row[0] for row in cursor)

print(f"Maximum centroid‑to‑point offset (meters): {max_dist}")

Maximum centroid‑to‑point offset (meters): 0.0020024987583835536


In [28]:
## calculate cell area of projected polygons

# add area field

Area = "Area_m2"

if Area not in [f.name for f in arcpy.ListFields(AOI_sqGrid_utm19n)]:
    arcpy.management.AddField(
        AOI_sqGrid_utm19n,
        Area,
        "DOUBLE"
    )

# calculate area using shape geometry

arcpy.management.CalculateGeometryAttributes(
    in_features=AOI_sqGrid_utm19n,
    geometry_property=[[Area, "AREA"]],
    area_unit="SQUARE_METERS"
)

<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb\\Orsted_sqGrid_utm19n'>

In [29]:
# validate calculations

areas = [row[0] for row in arcpy.da.SearchCursor(AOI_sqGrid_utm19n, [Area])]

print(f"Min area (m²): {min(areas)}")
print(f"Max area (m²): {max(areas)}")

# add area in sq km

arcpy.management.AddField(AOI_sqGrid_utm19n, "Area_km2", "DOUBLE")

arcpy.management.CalculateField(
    AOI_sqGrid_utm19n,
    "Area_km2",
    f"!{Area}! / 1e6",
    "PYTHON3"
)

Min area (m²): 928939.7760774809
Max area (m²): 937379.0488567835


<Result 'C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\GFW_VesselHotspots_SNE\\GFW_VesselHotspots_SNE.gdb\\Orsted_sqGrid_utm19n'>

In [30]:
# confirm grid cell consistency

import statistics

mean_area = statistics.mean(areas)
stdev_area = statistics.stdev(areas)

print(f"Mean area: {mean_area:.2f} m²")
print(f"Std deviation: {stdev_area:.2f} m²")

Mean area: 933160.35 m²
Std deviation: 2115.83 m²
